In [1]:
import arviz as az
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from jax.scipy.special import gammaln, logsumexp
from numpyro.infer import NUTS, MCMC, Predictive, SVI
import jax.scipy.special as jsp
import jax.scipy.stats as jsstats
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib_fontja

from numpyro.infer import SVI, Trace_ELBO
from numpyro.infer.autoguide import AutoDelta, AutoMultivariateNormal, AutoLowRankMultivariateNormal

In [2]:
import optuna
import pandas as pd
import random

In [3]:
def nb2_logsf_normal(cap_t, mu_t, alpha):
    var_t = mu_t + (mu_t**2) / alpha
    sigma_t = jnp.sqrt(var_t)
    return jsstats.norm.logsf(cap_t - 0.5, loc=mu_t, scale=sigma_t)

In [4]:
# Bass Model の定義
def bass_seasonal_censored_model_normal(
    t_values,
    season_indices,
    inventory_cap,
    sales_data=None,
    season_dim=12,
    **params,
):
    M = numpyro.sample("M", dist.LogNormal(params['M_mu'], params['M_var']))
    log_p = numpyro.sample("log_p", dist.Normal(params['log_p_mu'], params['log_p_var']))
    log_q = numpyro.sample("log_q", dist.Normal(params['log_q_mu'], params['log_q_var']))
    p = numpyro.deterministic("p", jnp.exp(log_p))
    q = numpyro.deterministic("q", jnp.exp(log_q))

    alpha = numpyro.sample("alpha", dist.Exponential(params['alpha_var']))
       
    concentration = numpyro.sample("conc", dist.LogNormal(0.0, params['season_sigma']))    
    # concentrationベクトルを作成（すべて同じ値にする場合）
    alpha_vec = jnp.ones(season_dim) * concentration    
    #  ディリクレ分布からサンプリング (合計は必ず 1.0 になる)
    s_raw = numpyro.sample("season_prob", dist.Dirichlet(alpha_vec))
    season_factors = numpyro.deterministic("season_factors", s_raw * season_dim)

    # ===============================
    # 2. Bass（連続時間・フロー）
    # ===============================
    m = p + q
    exp_term = jnp.exp(-m * t_values)

    def get_F(t):
        m = p + q
        exp_term = jnp.exp(-m * t)
        return (1.0 - exp_term) / (1.0 + (q / p) * exp_term)    
    
    prob_adoption_cumulative = get_F(t_values + 1.0) # tの終わり
    prob_adoption_prev = get_F(t_values)
    incr_prob = prob_adoption_cumulative - prob_adoption_prev
    
    # ベースの需要予測
    bass_flow = M * incr_prob

    # ===============================
    # 3. 季節性を掛けた平均需要（mu）
    # ===============================
    # year1_boost = numpyro.sample("year1_boost", dist.HalfNormal(params['boost']))
   
    # 2. ダミー変数の作成
    # t が 0~11 (1年目) なら 1.0, それ以降は 0.0
    # ※ t_values の単位に合わせて調整してください
    # is_first_year = (t_values < params['boost_period']).astype(jnp.float32)
    
    # 3. 乗数（Multiplier）の作成
    # 1年目は (1 + boost), 2年目以降は (1 + 0) = 1
    # shock_multiplier = 1.0 + year1_boost * is_first_year    
    
    mu_raw = bass_flow * season_factors[season_indices] # * shock_multiplier
    mu_t = jnp.clip(mu_raw, a_min=1e-9)
    numpyro.deterministic("demand_pred", mu_t)    

    # ===============================
    # 4. 尤度（NB2 + 右打ち切り：正規近似）
    # ===============================
    if sales_data is not None:
        y = jnp.asarray(sales_data).astype(jnp.int32)

        nb = dist.NegativeBinomial2(mean=mu_t, concentration=alpha)

        # 打ち切り判定（capが時系列）
        is_censored = y >= inventory_cap

        # 非打ち切り：通常のNB2
        logp_obs = nb.log_prob(y)

        # 打ち切り：log P(Y >= cap_t) を正規近似
        logp_cens = nb2_logsf_normal(inventory_cap, mu_t, alpha)

        pointwise_log_lik = jnp.where(is_censored, logp_cens, logp_obs)        
        numpyro.deterministic("log_likelihood", pointwise_log_lik)
        numpyro.factor("obs_loglik", jnp.sum(pointwise_log_lik))

In [5]:
df = pd.read_csv("data/20251101coolrevolution実績.csv", parse_dates=["受注_伝票登録日付"])
df = df.rename(columns={"受注_伝票登録日付": "date", "売上金額": "sales"})
df["day"] = df.date.dt.date
sales = df.groupby(pd.Grouper(key='date', freq='MS')).sales.size()
inventory_cap = pd.read_csv("data/クーレボ在庫.csv")
inventory_cap.yyyymm = inventory_cap.yyyymm.map(lambda x: pd.to_datetime(str(x)+"01"))
inventory_cap = inventory_cap.rename(columns={"yyyymm": "date", '在庫数（月末時点）': 'inventory'})
inventory_cap = inventory_cap.set_index('date')["inventory"]

data = pd.DataFrame({"sales": sales, "inventory": inventory_cap}).dropna(axis=0, how='any')
# data = data.reindex(index=pd.date_range("2019-10-01", '2025-06-01', freq='MS'))
data['month'] = data.index.month
data['inventory_cap'] = data[['inventory', 'sales']].max(axis=1)
data = data.fillna(0)
data = data.reset_index()

t_values = data.index
inv_cap = data['inventory_cap']
season_indices = data['month'] - 1
sales = data['sales']

In [6]:
# np.exp(5)

In [7]:
def run_optuna_tuning(
    t_values, 
    season_indices, 
    inventory_cap, 
    sales_data, 
    n_trials=20
):
    """
    Optunaを使ってBassモデルのハイパーパラメータ（Kや事前分布のスケール）を最適化する関数
    """
    
    # JAXのデータを準備（毎回変換しないように外でやる）
    j_t = jnp.asarray(t_values)
    j_season = jnp.asarray(season_indices)
    j_cap = jnp.asarray(inventory_cap)
    j_sales = jnp.asarray(sales_data)

    def objective(trial):
        # ==========================================
        # 1. Optunaによるハイパーパラメータの提案
        # ==========================================
               
        M_mu = trial.suggest_float("M_mu", 6.0, 8)
        M_var = trial.suggest_float("M_var", 0.1, 0.5)
        
        log_p_mu = trial.suggest_float("log_p_mu",  jnp.log(1e-3), jnp.log(0.2))
        log_p_var = trial.suggest_float("log_p_var", 0.01, 0.4, log=True)
        
        log_q_mu = trial.suggest_float("log_q_mu", jnp.log(0.05), jnp.log(1.0))
        log_q_var = trial.suggest_float("log_q_var", 0.01, 0.4, log=True)

        alpha_var = trial.suggest_float("alpha_var", 0.01, 0.4, log=True)
        season_sigma = trial.suggest_float("season_sigma", 0.01, 2.0, log=True)
#        boost = trial.suggest_float("boost", 0.01, 2, log=True)
#        boost_period = trial.suggest_int("boost_period", 0, 12)

        # ==========================================
        # 2. パラメータ辞書の構築
        # ==========================================
        model_kwargs = {
            "season_dim": 12,  # データに合わせて変更してください
            "M_mu": M_mu,
            "M_var": M_var,
            "alpha_var": alpha_var,
            "log_p_mu": log_p_mu,    # log変換済み
            "log_p_var": log_p_var,
            "log_q_mu": log_q_mu,    # log変換済み
            "log_q_var": log_q_var,            
            "season_sigma": season_sigma,
#            "boost": boost,
#            "boost_period": boost_period
        }

        # ==========================================
        # 2. MCMCの実行
        # ==========================================
        # 高速化のため、探索時は少しチェーン数やサンプル数を減らすのが定石
        try:
            kernel = NUTS(bass_seasonal_censored_model_normal, dense_mass=True)
            mcmc = MCMC(
                kernel, 
                num_warmup=300, 
                num_samples=2000, 
                progress_bar=False # ログが汚れるのでOFF推奨
            )
            
            # seedを変える必要がある場合は trial.number を使う
            rng_key = jax.random.PRNGKey(trial.number)
            
            mcmc.run(
                rng_key, 
                t_values=j_t, 
                season_indices=j_season, 
                inventory_cap=j_cap, 
                sales_data=j_sales, 
                **model_kwargs
            )
            
            # ==========================================
            # 3. WAICの計算
            # ==========================================
            # log_likelihoodが含まれていることを確認
            idata = az.from_numpyro(mcmc)
            waic_result = az.waic(idata, scale="deviance")            
            waic_score = waic_result.elpd_waic
            
            if jnp.isnan(waic_score) or jnp.isinf(waic_score):
                return float('inf')
                
            return waic_score
        except Exception as e:
            print(f"Trial {trial.number} failed: {e}")
            # エラーが出た場合は極端に悪い値を返して、このパラメータセットを避ける
            return float('inf')
    
    # ==========================================
    # 4. 最適化の実行
    # ==========================================
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)

    print("Best params:", study.best_params)
    print("Best WAIC:", study.best_value)
    
    return study

In [8]:
t_values = data.index
inv_cap = data['inventory_cap']
season_indices = data['month'] - 1
study = run_optuna_tuning(
    t_values=t_values.values, 
    season_indices=season_indices.values, 
    inventory_cap=inv_cap.values, 
    sales_data=sales.values, 
    n_trials=300
)

[I 2025-12-15 21:07:03,811] A new study created in memory with name: no-name-e8f3be72-2911-49fd-bd3f-e7a9aa971200
/Users/tsuyos-u/Library/Caches/pypoetry/virtualenvs/semi-bayes-tseries-wVQtb5VF-py3.11/lib/python3.11/site-packages/arviz/stats/stats.py:1667: UserWarning: For one or more samples the posterior variance of the log predictive densities exceeds 0.4. This could be indication of WAIC starting to fail. 
See http://arxiv.org/abs/1507.04544 for details
  warnings.warn(
/Users/tsuyos-u/Library/Caches/pypoetry/virtualenvs/semi-bayes-tseries-wVQtb5VF-py3.11/lib/python3.11/site-packages/arviz/stats/stats.py:1695: UserWarning: The point-wise WAIC is the same with the sum WAIC, please double check
            the Observed RV in your model to make sure it returns element-wise logp.
            
  warnings.warn(
[I 2025-12-15 21:07:10,073] Trial 0 finished with value: 596.495849609375 and parameters: {'M_mu': 6.520754908151377, 'M_var': 0.11678574131403147, 'log_p_mu': -6.640146238377356,

Best params: {'M_mu': 7.844583378917458, 'M_var': 0.4062665459054293, 'log_p_mu': -5.146741193561419, 'log_p_var': 0.2547433338529219, 'log_q_mu': -2.891575493012795, 'log_q_var': 0.3983405034525006, 'alpha_var': 0.02799360905889035, 'season_sigma': 0.03351216396323586}
Best WAIC: 328.8950443267822


In [9]:
study.best_params

{'M_mu': 7.844583378917458,
 'M_var': 0.4062665459054293,
 'log_p_mu': -5.146741193561419,
 'log_p_var': 0.2547433338529219,
 'log_q_mu': -2.891575493012795,
 'log_q_var': 0.3983405034525006,
 'alpha_var': 0.02799360905889035,
 'season_sigma': 0.03351216396323586}

In [10]:
import numpy as np
# np.log(5000)

In [11]:
# np.log(1e-6)